# ÍSKOS 2024 — 02. Líkön

Frá einföldu trénu til skógar. Við þjálfum þrjú líkön á vaxandi flækjustigi og berum saman:

1. **Dummy-grunnlína** — giskar alltaf á algengasta flokkinn. Gólfið sem flóknara líkan verður að slá.
2. **Ákvörðunartré (depth=3)** — túlkanlegt, sýnir reglur sem hægt er að lesa.
3. **Djúpt ákvörðunartré (depth=None)** — sýnir ofmátun.
4. **Slembiskógur (300 tré)** — meðaltal margra trjáa sem hvert lærir á tilviljanakenndu úrtaki af gögnum og breytum.

Eingöngu slembiskógurinn fær eigið ruglingsfylki og er vistaður — næsta vinnubók notar hann fyrir SHAP-útskýringar og þína eigin spá.

In [ ]:
import json
from pathlib import Path

import joblib
import pandas as pd
import matplotlib.pyplot as plt
import graphviz

import iskos_style as ik   # ÍSKOS litir + þema (iskos_style.py) — sami stíll og rsetup.R

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, export_graphviz  # noqa: F401
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay

PROJECT = Path.cwd().parent
DATA = PROJECT / "data"
MODELS = PROJECT / "models"; MODELS.mkdir(exist_ok=True)
FIGS = PROJECT / "figures"; FIGS.mkdir(exist_ok=True)

df = pd.read_csv(DATA / "iskos_clean.csv")
# Index meta by the Icelandic question text — same as df's column names.
meta = pd.read_csv(DATA / "feature_meta.csv").set_index("label")
ik.apply_rc()   # ÍSKOS bakgrunnur + letur fyrir allar myndir
print(f"{len(df)} svarendur · {df.shape[1] - 1} breytur")

## 1. Sleppum leakage-hópunum

Hópur **4 (Málefnaeign)** og hópur **5 (Flokkshollusta og leiðtogar)** spyrja í raun *„hver er flokkurinn þinn?“* með óbeinum hætti — t.d. *„hvaða flokkur er bestur í efnahagsmálum?“* eða *„hversu vel líkar þér við Sjálfstæðisflokkinn (0–10)?“*. Ef við setjum þessar breytur inn í líkanið mun nákvæmnin hækka — en það er **leakage**, ekki forspá.

Sama gildir að hluta um **mat á frammistöðu ríkisstjórnarinnar** (`govprfm`). Ríkisstjórnin 2021–2024 var skipuð VG, Sjálfstæðisflokki og Framsóknarflokki, svo spurningin *„hversu vel stóð ríkisstjórnin sig?“* er nátengd því hvort svarandi kaus stjórnarflokk. Við sleppum henni því líka, eins og gert er í kaflanum í bókinni.

**Prufaðu sjálf(ur):** breyttu `LEAK_GROUPS` og/eða `EXCLUDE_VARS` í tóma sett-ið `set()` og keyrðu vinnubókina aftur. Nákvæmni slembiskógarins hækkar verulega þegar leakage-hóparnir eru teknir með — það er gervihækkun sem byggir á eiginleikum sem eru of nálægt markmiðinu.

In [ ]:
LEAK_GROUPS = {"4 – Málefnaeign", "5 – Flokkshollusta og leiðtogar"}
# LEAK_GROUPS = set()   # ← uncomment to include leakage groups and watch accuracy jump

# Single variables left out for the same reason: evaluation of the outgoing VG–D–B
# government is closely tied to whether the respondent voted for a governing party.
EXCLUDE_VARS = {"govprfm"}
# EXCLUDE_VARS = set()   # ← uncomment to put government performance back in

leak_vars = meta.index[
    meta["group"].isin(LEAK_GROUPS) | meta["short_name"].isin(EXCLUDE_VARS)
].tolist()
print(f"Sleppum {len(leak_vars)} breytum: hópum {LEAK_GROUPS} og breytunum {EXCLUDE_VARS}")
X_raw = df.drop(columns=["flokkur"] + leak_vars)
y = df["flokkur"]
print("X_raw:", X_raw.shape)

## 2. One-hot kóðun — vinnandi dæmi: `gender`

Slembiskógar í sklearn taka aðeins við tölum, ekki strengjum. Þess vegna umbreytum við hverri **flokkabreytu** í einn dálk á hvert mögulegt gildi (0 eða 1). Raðbreytur halda sér óbreyttar.

In [ ]:
# Look up the Icelandic question text for the short code `gender`.
gender_q = meta[meta["short_name"] == "gender"].index[0]
pd.get_dummies(X_raw[[gender_q]]).head()

## 3. One-hot á öllum eiginleikum + skipting í þjálfunar- og prufuúrtak

`pd.get_dummies` snertir bara strengjadálka. Eftir kóðunina deilum við 80/20 í `train`/`test` úrtök með `stratify=y` til að halda hlutföllum flokkanna jöfnum, og `random_state=42` til að fá endurtakanlegan kóða.

In [ ]:
X = pd.get_dummies(X_raw)
print("Eftir one-hot:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## 4. Grunnlína: algengasti flokkurinn

Gólfið — `DummyClassifier(strategy="most_frequent")` giskar alltaf á Samfylkinguna (algengasta flokkinn í þjálfunarúrtakinu). Nákvæmnin á prufuúrtaki er einfaldlega hlutfall þess flokks í gögnunum.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Dummy test accuracy: {dummy.score(X_test, y_test):.3f}")

## 5. Ákvörðunartré (max_depth=3)

Grunnt tré er einfaldasta líkanið sem getur lært mörg mismunandi mynstur. Þú getur lesið reglurnar beint af myndinni hér að neðan: hver hnútur sýnir **klofningsreglu**, **algengasta flokkinn** í þeim hluta og **fjölda svarenda** sem þangað ná.

In [ ]:
dt = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print(f"Grunnt tré – train: {dt.score(X_train, y_train):.3f}  test: {dt.score(X_test, y_test):.3f}")

### Trésmyndin

Við sleppum stöðluðu sklearn-myndinni (sem sýnir `gini`, `samples`, `value=[…]`) og smíðum okkar eigin graphviz-mynd með aðeins þrennt í hverjum hnút: **reglu, flokk, n=…**. Það er nóg til að skilja hvað líkanið er að gera.

In [ ]:
# Flokkalitir koma nú úr iskos_style.py (party_colors / class_colors) — sömu mettuðu
# vörumerkjalitir og í rsetup.R. Þar sem litirnir eru dökkir veljum við texta (svartan/hvítan)
# eftir birtu hvers litar með ik.text_on(), svo textinn í hverjum hnút sé alltaf læsilegur.
PARTY_COLORS = ik.CLASS_COLORS
# One-letter party code (last char before the closing paren in the class label).
PARTY_LETTER = {
    "Samfylkinguna (S)":          "S",
    "Sjálfstæðisflokkinn (D)":    "D",
    "Viðreisn (C)":               "C",
    "Framsóknarflokkinn (B)":     "B",
    "Miðflokkinn (M)":            "M",
    "Flokk fólksins (F)":         "F",
    "Vinstri (sameinað)":         "V",
}

# Load short_label lookup from feature_meta so split text under each node fits cleanly.
import math
_meta_for_tree = pd.read_csv(DATA / "feature_meta.csv").set_index("label")
# Load value_labels.json so we can show the Likert text (e.g. "Hlutlaus/óviss") next to
# the integer threshold for ordinal questions.
with open(DATA / "value_labels.json", encoding="utf-8") as _f:
    _VALUE_LABELS = json.load(_f)

def _short_for_split(col):
    """Map a model column name (possibly one-hot encoded) to a readable short form.
    Examples: 'Eigin staðsetning á vinstri-hægri kvarða' → 'Vinstri-hægri staða';
              'Kjördæmi_Suðvesturkjördæmi' → 'Kjördæmi = Suðvesturkjördæmi'."""
    if "short_label" not in _meta_for_tree.columns:
        return col
    if col in _meta_for_tree.index:
        s = _meta_for_tree.loc[col, "short_label"]
        return s.strip() if isinstance(s, str) and s.strip() else col
    for lbl in _meta_for_tree.index:
        if col.startswith(lbl + "_"):
            value = col[len(lbl) + 1:]
            s = _meta_for_tree.loc[lbl, "short_label"]
            short = s.strip() if isinstance(s, str) and s.strip() else lbl
            return f"{short} = {value}"
    return col


def _root_label_for_col(col):
    """Return the original (unsuffixed) variable label for a model column, or None for one-hots."""
    if col in _meta_for_tree.index:
        return col
    return None  # one-hot dummies (e.g. 'Kjördæmi_Suðvesturkjördæmi') have no Likert text


def _split_rule_text(col, thr):
    """Render a sklearn `feature <= threshold` rule as an integer-clean human string.

    Since every feature in this dataset is integer-coded (Likert 1–5, 0–10 etc.),
    a threshold of x.5 is equivalent to a strict integer comparison:
        feature <= 6.5   ⟺   feature < 7
    For ordinal Likert variables we additionally append the value label of the
    boundary integer, so a reader doesn't have to remember what e.g. 4 means.
    """
    short = _short_for_split(col)
    # Most sklearn thresholds for integer data are exactly x.5; if it's already an
    # integer (rare here) keep the <= form so we don't shift the partition.
    if abs(thr - round(thr)) < 1e-9:
        int_thr = int(round(thr))
        rule = f"{short} ≤ {int_thr}"
    else:
        int_thr = int(math.ceil(thr))
        rule = f"{short} < {int_thr}"

    root = _root_label_for_col(col)
    if root is not None and root in _VALUE_LABELS:
        text = _VALUE_LABELS[root].get(str(int_thr))
        # Skip labels that are just the digit (e.g. "7" on the 0–10 scale) — they add no info.
        if text and text.strip() != str(int_thr):
            rule = f"{rule} ({text})"
    return rule


def render_tree(clf, feature_names, class_names, out_basename: Path):
    """Render a sklearn DecisionTreeClassifier with party-themed colours.

    Each node shows:
        <party letter>
        D:n  S:n  B:n  C:n  F:n  M:n  V:n   (per-class counts in label order)
        XX%                                  (proportion of total observations)
        <split rule>                         (only for non-leaf nodes — at the bottom)
    """
    tree = clf.tree_
    total_n = int(tree.n_node_samples[0])
    letters = [PARTY_LETTER.get(c, "?") for c in class_names]

    lines = [
        "digraph Tree {",
        # ranksep eykur lóðrétt bil milli hæða svo myndin verði hærri og textinn læsilegur.
        '  graph [fontname="Arial", dpi=170, ranksep=0.6, nodesep=0.3];',
        '  node [shape=box, style="rounded,filled", fontname="Arial", fontsize=12, margin="0.18,0.10"];',
        '  edge [fontname="Arial", fontsize=11];',
    ]
    for i in range(tree.node_count):
        # sklearn ≥1.4 stores `value` as class proportions (sum to 1); reconstruct integer counts.
        props = tree.value[i].squeeze()
        n = int(tree.n_node_samples[i])
        counts = (props * n).round().astype(int)
        majority_idx = int(props.argmax())
        majority = class_names[majority_idx]
        maj_letter = PARTY_LETTER.get(majority, "?")
        is_leaf = tree.children_left[i] == -1

        count_line = "  ".join(f"{letters[k]}:{counts[k]}" for k in range(len(class_names)))
        pct = 100.0 * n / total_n

        if is_leaf:
            label = f"{maj_letter}\n{count_line}\n{pct:.0f}%"
        else:
            rule_text = _split_rule_text(feature_names[tree.feature[i]], tree.threshold[i])
            # Wrap if long — keep boxes from blowing up.
            if len(rule_text) > 36:
                words = rule_text.split()
                wrapped, cur = [], ""
                for w in words:
                    if cur and len(cur) + 1 + len(w) > 36:
                        wrapped.append(cur); cur = w
                    else:
                        cur = f"{cur} {w}".strip()
                if cur:
                    wrapped.append(cur)
                rule_text = "\n".join(wrapped)
            label = f"{maj_letter}\n{count_line}\n{pct:.0f}%\n{rule_text}"

        color = PARTY_COLORS.get(majority, "#999999")
        fontcolor = ik.text_on(color)   # svartur/hvítur texti eftir birtu flokkslitarins
        lines.append(f'  {i} [label="{label}", fillcolor="{color}", fontcolor="{fontcolor}"];')
        if not is_leaf:
            lines.append(f'  {i} -> {tree.children_left[i]} [label="já"];')
            lines.append(f'  {i} -> {tree.children_right[i]} [label="nei"];')
    lines.append("}")
    src = graphviz.Source("\n".join(lines))
    src.format = "png"
    src.render(filename=out_basename.name, directory=str(out_basename.parent), cleanup=True)
    return out_basename.with_suffix(".png")

tree_png = render_tree(
    dt,
    feature_names=X.columns.tolist(),
    class_names=dt.classes_.tolist(),
    out_basename=FIGS / "decision_tree_depth3",
)
from IPython.display import Image, display
display(Image(filename=str(tree_png)))


## 6. Of djúpt tré — ofmátun

Ef við leyfum trénu að vaxa óheft (`max_depth=None`), lærir það þjálfunargögnin **utan að**: train-nákvæmni fer í 100%, en test-nákvæmni versnar. Þetta er **ofmátun** og er kjarnaástæðan fyrir því að við notum slembiskóga í staðinn.

In [ ]:
dt_deep = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print("           train    test")
print(f"Grunnt:  {dt.score(X_train, y_train):.3f}   {dt.score(X_test, y_test):.3f}")
print(f"Djúpt:   {dt_deep.score(X_train, y_train):.3f}   {dt_deep.score(X_test, y_test):.3f}")

## 7. Slembiskógur

Slembiskógur er meðaltal af 300 trjám, þar sem hvert tré er þjálfað á tilviljanakenndu úrtaki af bæði svarendum og eiginleikum. Hvert tré ofmátar á sinn hátt — meðaltal af mörgum ofmátuðum trjám er mun stöðugra en eitt tré.

Þetta er líkanið sem þú þjálfar í alvöruverkefnum þegar þú vilt eitthvað sterkt án mikillar fínstillingar.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
).fit(X_train, y_train)
print(f"Random forest – train: {rf.score(X_train, y_train):.3f}  test: {rf.score(X_test, y_test):.3f}")

### Top-k nákvæmni — hve oft er rétt svar á topp-2 listanum?

Stundum þarf líkanið ekki að hitta nákvæmlega rétt á flokk til að vera gagnlegt — t.d. ef við viljum bara segja hvaða flokka kjósandi kaus *ekki*. `Top-2 accuracy` mælir hve oft raunverulegt flokksval svaranda er meðal tveggja líklegustu spáðu flokkanna.

In [ ]:
import numpy as np

probs_test = rf.predict_proba(X_test)             # n_test x n_classes
class_names = rf.classes_

# For each row, take the top-2 class indices by probability and check if true class is among them.
top2_idx = np.argsort(-probs_test, axis=1)[:, :2]
y_test_arr = y_test.values
top2_correct = np.array([y_test_arr[i] in class_names[top2_idx[i]]
                          for i in range(len(y_test_arr))])

top1_acc = (rf.predict(X_test) == y_test).mean()
top2_acc = top2_correct.mean()
print(f"Top-1 nákvæmni:  {top1_acc:.3f}")
print(f"Top-2 nákvæmni:  {top2_acc:.3f}")


### Jafnvægisstilling — `class_weight="balanced_subsample"`

Flokkarnir í úrtakinu eru misstórir: Samfylkingin er með ~250 svarendur en Framsóknarflokkurinn aðeins ~93. Líkanið lærir því náttúrulega að *spá frekar Samfylkinguna*, því það er „öruggara“ — það er meira að græða á réttum stærsta flokki en á réttum litlum flokki.

Sklearn-skjölun tilgreinir nokkra möguleika fyrir `class_weight`:

- `"balanced"` — reiknar vægi á öllu þjálfunarúrtakinu sem `n_samples / (n_classes * np.bincount(y))`.
- `"balanced_subsample"` — sama, **en endurreiknað á bootstrap-úrtaki hvers trés**. Mælt með fyrir slembiskóga því hvert tré sér ekki nákvæmlega sömu flokkadreifingu og heildarúrtakið.

Við notum `"balanced_subsample"`. Niðurstöðurnar fyrir neðan sýna að heimt (e. recall) eykst örlítið með því að nota þessa jafnvægisstillingu.

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
).fit(X_train, y_train)
print(f"Slembiskógur (jafnvægisstilling) – "
      f"train: {rf_balanced.score(X_train, y_train):.3f}  "
      f"test: {rf_balanced.score(X_test, y_test):.3f}")

# Compare per-class recall — the metric that imbalance correction is supposed to help.
from sklearn.metrics import recall_score
for name, model in [("naive", rf), ("balanced", rf_balanced)]:
    recalls = recall_score(y_test, model.predict(X_test),
                           labels=model.classes_, average=None, zero_division=0)
    macro = recalls.mean()
    print(f"\n{name:>9s}:  macro-recall = {macro:.3f}")
    for c, r in zip(model.classes_, recalls):
        print(f"            {c:32s} recall={r:.2f}")

## 8. Ruglingsfylki slembiskógarins

Ruglingsfylki sýnir fyrir hvern raunverulegan flokk (röð/lína) hvernig líkanið flokkaði svarendurna (dálkur). Á hornalínunni eru réttar spár; reitir utan hornalínunnar eru ranglega flokkaðir.

Lestu þetta sem: *„þegar raunverulegir Samfylkingar-kjósendur eru í prófunarúrtakinu, hvernig spáir líkanið um þá?“*. Skoðaðu hvernig líkanið ruglar flokkunum saman.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
# Litabrú hvítt → iskos_color_main svo ruglingsfylkið noti meginlit bókarinnar.
iskos_blues = LinearSegmentedColormap.from_list("iskos_blues", ["#ffffff", ik.ISKOS_MAIN])

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ConfusionMatrixDisplay.from_estimator(
    rf, X_test, y_test, ax=ax, cmap=iskos_blues, # Prófaðu að skipta rf út fyrir rf_balanced til að sjá ruglingsfylkið með jafnvægisstillingu
    xticks_rotation=45, colorbar=False,
)
ik.style_iskos(ax, title="Ruglingsfylki — slembiskógur")
plt.tight_layout()
plt.savefig(FIGS / "confusion_matrix_rf.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Mikilvægustu breyturnar

`feature_importances_` í slembiskógi mælir hversu mikið hver eiginleiki minnkar Gini stuðulinn að meðaltali yfir trén 300 (Gini stuðullinn segir hversu ójöfn dreifing flokkanna er). Það er gróf nálgun til að meta mikilvægi breyta. Næsta vinnubók notar **SHAP** sem gefur dýpri sundurliðun fyrir hvern flokk og hvern einstakling.

In [ ]:
# Aggregate one-hot column importance back to the root Icelandic question, then apply the
# short_label override so the bar chart is readable. Without this, the top 15 is dominated
# by individual one-hot columns like "Kjördæmi_Suðvesturkjördæmi", which is noisy.
def _root_for_imp(col):
    for lbl in meta.index:
        if col == lbl or col.startswith(lbl + "_"):
            return lbl
    return col

def _short_for_imp(name):
    if name in meta.index and "short_label" in meta.columns:
        s = meta.loc[name, "short_label"]
        if isinstance(s, str) and s.strip():
            return s.strip()
    return name

raw_imps = pd.Series(rf.feature_importances_, index=X.columns)
agg_imps = raw_imps.groupby(raw_imps.index.map(_root_for_imp)).sum().sort_values(ascending=True)
top = agg_imps.tail(15)
top.index = [_short_for_imp(n) for n in top.index]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index, top.values, color=ik.ISKOS_MAIN)   # iskos_color_main
ax.set_xlabel("Mikilvægi (Gini importance)")
ik.style_iskos(ax, title="15 mikilvægustu eiginleikar slembiskógarins")
plt.tight_layout()
plt.show()


## 10. Vistum líkanið

Til þess að næsta vinnubók geti spáð fyrir um þína eigin svör þurfum við að vista tvennt:

- `rf.pkl` — þjálfaða slembiskóginn.
- `feature_columns.json` — listinn af eiginleikum eftir one-hot kóðun, í nákvæmlega þeirri röð sem líkanið var þjálfað á. Þetta er kritískt: einnar línu DataFrame frá þér þarf að passa við nákvæmlega þessa dálkaröð, annars spáir líkanið vitleysu.

In [ ]:
joblib.dump(rf, MODELS / "rf.pkl")
with open(MODELS / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(X.columns.tolist(), f, ensure_ascii=False, indent=2)
print(f"Vistaði líkan: {(MODELS / 'rf.pkl').relative_to(PROJECT)}")
print(f"Vistaði dálkaröð: {(MODELS / 'feature_columns.json').relative_to(PROJECT)} ({X.shape[1]} dálkar)")